# Meridian: lumbar-spine patient pathway timing

This is a standalone EDA notebook for the clinical journey described in the client brief: **earliest record → imaging → referral → injections → surgical consultation → surgery**. It uses the curated orthopedic encounters table to define clinical milestones, and uses claims and coverage only to establish the first observable record.

# Part H: Patient pathway timing

**Clinical-stage rule:** a milestone is the first `Performed` row in the curated encounter category. Scheduled, cancelled, and no-show rows are excluded because they do not document a completed clinical event. The category mapping assigns every CPT and ICD-10-CM code in this curated extract; code-missing rows remain visible because their category still identifies the event.

**Earliest-record rule:** the earliest of coverage start, any encounter, and any claim. This is an observation-start proxy, not onset of low-back pain.

Each matrix cell is `first column event − first row event` in days, computed among patients observed at both events. Negative values are retained: they indicate an out-of-order observed pathway or incomplete history. Counts are pair-specific, so the analysis does not require patients to complete all six stages.

In [1]:
import numpy as np
import pandas as pd

STAGES = ['Earliest record', 'Imaging', 'Referred', 'Injections', 'See surgeon', 'Surgery']
CATEGORY_TO_STAGE = {
    'LBP Encounter': 'Earliest record',
    'Imaging': 'Imaging',
    'PT Referral': 'Referred',
    'Injection': 'Injections',
    'Surgical Consult': 'See surgeon',
    'Spine Surgery': 'Surgery',
}

encounters = pd.read_csv('../data/meridian_health_encounters.csv', dtype=str)
encounters['event_date'] = pd.to_datetime(encounters['event_date'], errors='coerce')
encounters['pathway_stage'] = encounters['event_category'].map(CATEGORY_TO_STAGE)

# Explicit taxonomy: all curated CPT/ICD codes and the category they represent.
code_taxonomy = (encounters.groupby(['pathway_stage', 'event_category', 'code_system', 'event_code'],
                                   dropna=False, observed=True)
                 .agg(events=('event_id', 'size'), patients=('patient_token', 'nunique'))
                 .reset_index()
                 .rename(columns={'event_code': 'code', 'code_system': 'system'}))
code_reference = pd.read_csv('../data/cpt_icd10_codes.csv', dtype=str)[['Code', 'Name', 'Description']]
code_taxonomy = (code_taxonomy.merge(code_reference, left_on='code', right_on='Code', how='left')
                 .drop(columns='Code'))
code_taxonomy['pathway_stage'] = pd.Categorical(code_taxonomy['pathway_stage'], STAGES, ordered=True)
code_taxonomy = code_taxonomy.sort_values(['pathway_stage', 'system', 'code'], na_position='last').reset_index(drop=True)

# First observable date: coverage start plus broad claims and curated encounters.
patients = pd.read_csv('../data/meridian_health_patients.csv', usecols=['patient_token', 'coverage_start'], dtype=str)
patients['coverage_start'] = pd.to_datetime(patients['coverage_start'], errors='coerce')
claims = pd.read_csv('../data/data_web_claims.csv', usecols=['patient_token', 'service_date'], dtype=str)
claims['service_date'] = pd.to_datetime(claims['service_date'], errors='coerce')
record_dates = pd.concat([
    patients.rename(columns={'coverage_start': 'date'})[['patient_token', 'date']],
    encounters.rename(columns={'event_date': 'date'})[['patient_token', 'date']],
    claims.rename(columns={'service_date': 'date'})[['patient_token', 'date']],
])
first_record = (record_dates.dropna(subset=['date']).groupby('patient_token')['date'].min()
                .rename('Earliest record'))

# Completed clinical milestones only. LBP Encounter is classified for the taxonomy but does not override broad first record.
pathway_events = encounters[encounters['pathway_stage'].notna() & encounters['event_status'].eq('Performed')].copy()
first_clinical = (pathway_events[pathway_events['pathway_stage'].ne('Earliest record')]
                  .groupby(['patient_token', 'pathway_stage'], observed=True)['event_date'].min()
                  .unstack())
milestones = first_record.to_frame().join(first_clinical, how='left').reindex(columns=STAGES)

# Pairwise timing and patient counts. Row is the source event; column is the target event.
long_milestones = (milestones.rename_axis('patient_token').reset_index()
                   .melt('patient_token', var_name='stage', value_name='date')
                   .dropna(subset=['date']))
pairs = long_milestones.merge(long_milestones, on='patient_token', suffixes=('_from', '_to'))
pairs['days'] = (pairs['date_to'] - pairs['date_from']).dt.days
pair_summary = (pairs.groupby(['stage_from', 'stage_to'], observed=True)['days']
                .agg(mean_days='mean', median_days='median', n='size').reset_index())

def pathway_matrix(metric):
    return (pair_summary.pivot(index='stage_from', columns='stage_to', values=metric)
            .reindex(index=STAGES, columns=STAGES))

mean_days = pathway_matrix('mean_days')
median_days = pathway_matrix('median_days')
pair_counts = pathway_matrix('n').astype('Int64')

adjacent_steps = []
for stage_from, stage_to in zip(STAGES[:-1], STAGES[1:]):
    gaps = pairs.loc[(pairs['stage_from'] == stage_from) & (pairs['stage_to'] == stage_to), 'days']
    adjacent_steps.append({
        'from': stage_from, 'to': stage_to, 'patients': len(gaps),
        'mean_days': gaps.mean(), 'median_days': gaps.median(),
        'observed_in_expected_order': (gaps >= 0).mean(),
    })
adjacent_summary = pd.DataFrame(adjacent_steps)


## Code taxonomy and adjacent-step summary

In [2]:
display(code_taxonomy)
display(adjacent_summary.style.format({
    'mean_days': '{:,.1f}',
    'median_days': '{:,.1f}',
    'observed_in_expected_order': '{:.1%}',
}))

,pathway_stage,event_category,system,code,events,patients,Name,Description
0,Earliest record,LBP Encounter,ICD10CM,G89.29,16586,12008,Other chronic pain,Chronic pain not classified elsewhere; often p...
1,Earliest record,LBP Encounter,ICD10CM,M43.16,16730,12119,"Spondylolisthesis, lumbar region",Forward slippage of one lumbar vertebra over t...
2,Earliest record,LBP Encounter,ICD10CM,M48.061,16212,11925,"Spinal stenosis, lumbar, without neurogenic cl...",Narrowed lumbar canal without walking-induced ...
3,Earliest record,LBP Encounter,ICD10CM,M48.062,16530,12049,"Spinal stenosis, lumbar, with neurogenic claud...",Narrowed lumbar canal causing leg pain/weaknes...
4,Earliest record,LBP Encounter,ICD10CM,M51.36,16505,12070,"Other intervertebral disc degeneration, lumbar...",Degenerative disc disease of the lumbar spine
5,Earliest record,LBP Encounter,ICD10CM,M54.16,16541,12093,"Radiculopathy, lumbar region","Nerve root pain, numbness or weakness from the..."
6,Earliest record,LBP Encounter,ICD10CM,M54.17,16300,11930,"Radiculopathy, lumbosacral region",Nerve root symptoms arising at the lumbosacral...
7,Earliest record,LBP Encounter,ICD10CM,M54.50,16512,12037,"Low back pain, unspecified",Lumbago with no further specificity
8,Earliest record,LBP Encounter,ICD10CM,M54.51,16473,12017,Vertebrogenic low back pain,Back pain originating from the vertebral endpl...
9,Earliest record,LBP Encounter,ICD10CM,M54.59,16585,12041,Other low back pain,Low back pain of a specified type not covered ...


,from,to,patients,mean_days,median_days,observed_in_expected_order
0,Earliest record,Imaging,17419,571.7,569.0,100.0%
1,Imaging,Referred,8053,16.6,1.0,50.3%
2,Referred,Injections,3086,66.7,71.0,62.7%
3,Injections,See surgeon,1363,-230.9,-187.0,12.7%
4,See surgeon,Surgery,765,168.6,168.0,92.7%


## Average and median elapsed days between milestones

Read each cell from its row to its column. The annotation includes its own denominator (`n`).

In [3]:
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), constrained_layout=True)
values = np.concatenate([mean_days.to_numpy().ravel(), median_days.to_numpy().ravel()])
limit = np.nanmax(np.abs(values))

for ax, matrix, label in zip(axes, [mean_days, median_days], ['Average days', 'Median days']):
    image = ax.imshow(matrix, cmap='RdBu_r', vmin=-limit, vmax=limit)
    ax.set_xticks(range(len(STAGES)), STAGES, rotation=35, ha='right')
    ax.set_yticks(range(len(STAGES)), STAGES)
    ax.set_xlabel('Target milestone (column)')
    ax.set_ylabel('Source milestone (row)')
    ax.set_title(label)
    for row in range(len(STAGES)):
        for col in range(len(STAGES)):
            days, n = matrix.iat[row, col], pair_counts.iat[row, col]
            if pd.notna(days):
                text_color = 'white' if abs(days) > 0.55 * limit else '#222222'
                ax.text(col, row, f'{days:,.0f} d\nn={n:,}', ha='center', va='center',
                        fontsize=8, color=text_color)
    fig.colorbar(image, ax=ax, shrink=0.80, label='Days (target − source)')

fig.suptitle('Observed lumbar-spine pathway: time between first completed milestones', fontsize=14)
plt.show()

/var/folders/ty/694s8g7n1cz_l5s3prx55jzr0000gn/T/ipykernel_11786/3610542944.py:25: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Interpretation guardrails

- The extract begins on 2023-07-01, so early pathway dates may be left-truncated.
- The matrices describe observed care ordering, not a causal sequence. In particular, a negative gap can reflect care outside this service-line extract, delayed documentation, or truly different treatment order.
- The broad claims feed is deliberately not used to infer imaging, referral, injection, consultation, or surgery because several codes are not spine-specific outside the curated encounter context.